# 📈 Nifty 50 ATM Options — Monthly Expiry Backtest

**Strategies compared:** SMA-9 vs EMA-9 on monthly expiry buy-side options

**Notebook flow:**
1. Clone repo & install dependencies
2. Load & inspect Nifty 50 EOD data
3. Feature engineering (MAs, ATR, expiry flags)
4. Run backtest for both strategies
5. Analyse results — metrics, equity curves, trade log
6. Deep-dive charts & export

## 0 · Setup — Clone Repo & Install

In [ ]:
# ── Clone the repo (only needed first time) ──────────────
import os

REPO_URL = "https://github.com/YOUR_USERNAME/nifty-options-backtest.git"  # ← update this
REPO_DIR = "nifty-options-backtest"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}

%cd {REPO_DIR}

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import sys, os
sys.path.insert(0, '.')   # so 'src' package resolves

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display

from src.backtest import (
    NiftyDataLoader,
    DataEngineering,
    SMA9Strategy,
    EMA9Strategy,
    Backtester,
)

pd.set_option('display.float_format', '{:,.2f}'.format)
plt.rcParams['figure.dpi'] = 110
print('✅ Imports OK')

## 1 · Load Nifty 50 EOD Data

In [ ]:
# ── Config ────────────────────────────────────────────────
YEARS          = 5       # how many years of data to backtest
INITIAL_CAPITAL = 500_000  # ₹5 lakh

# ── Load ─────────────────────────────────────────────────
loader = NiftyDataLoader(years=YEARS)
df_raw = loader.load()

print(f"\nShape : {df_raw.shape}")
df_raw.tail()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4), facecolor='#0d1117')
ax.set_facecolor('#161b22')
ax.plot(df_raw.index, df_raw['Close'], color='#00d4aa', linewidth=1)
ax.set_title('Nifty 50 — Closing Price', color='white', fontsize=13)
ax.set_ylabel('Close (₹)', color='#aaa')
ax.tick_params(colors='#777')
for sp in ax.spines.values(): sp.set_edgecolor('#333')
plt.tight_layout()
plt.show()

## 2 · Feature Engineering

In [ ]:
eng = DataEngineering(df_raw)
df  = eng.build()

print("\nFeature columns:", df.columns.tolist())
df.head(3)

In [ ]:
# Inspect identified monthly expiry dates
expiry_df = df[df['Monthly_Expiry']][['Close', 'SMA_9', 'EMA_9', 'ATR_14', 'ATM_Strike']]
print(f"Total monthly expiry dates found: {len(expiry_df)}")
expiry_df.tail(10)

In [ ]:
# Plot price vs SMA/EMA with expiry markers
fig, ax = plt.subplots(figsize=(14, 5), facecolor='#0d1117')
ax.set_facecolor('#161b22')

last = df.last('365D')  # last 1 year for clarity
ax.plot(last.index, last['Close'],  color='#7eb8f7', linewidth=1,   label='Close')
ax.plot(last.index, last['SMA_9'],  color='#f7931e', linewidth=1.2, label='SMA-9', linestyle='--')
ax.plot(last.index, last['EMA_9'],  color='#00d4aa', linewidth=1.2, label='EMA-9', linestyle=':')

# Mark expiry dates
exp = last[last['Monthly_Expiry']]
ax.scatter(exp.index, exp['Close'], color='#e05c7f', zorder=5, s=40, label='Expiry')

ax.legend(facecolor='#222', labelcolor='white', fontsize=9)
ax.set_title('Nifty 50 — Close vs SMA-9 vs EMA-9 (last 1 year)', color='white')
ax.tick_params(colors='#777')
for sp in ax.spines.values(): sp.set_edgecolor('#333')
plt.tight_layout()
plt.show()

## 3 · Run Backtest

In [ ]:
bt = Backtester(df)
bt.INITIAL_CAPITAL = INITIAL_CAPITAL

bt.add_strategy(SMA9Strategy())
bt.add_strategy(EMA9Strategy())

bt.run()
bt.print_summary()

## 4 · Results Analysis

In [ ]:
# Tidy metrics table
metrics_df = bt.get_metrics_df()
display(metrics_df.T.style
    .format({
        'total_pnl'   : '₹{:,.0f}',
        'max_drawdown': '₹{:,.0f}',
        'avg_pnl'     : '₹{:,.0f}',
        'win_rate'    : '{:.1f}%',
        'roi_pct'     : '{:.1f}%',
    })
    .background_gradient(cmap='RdYlGn', axis=1)
)

In [ ]:
# Overlay equity curves
fig, ax = plt.subplots(figsize=(13, 5), facecolor='#0d1117')
ax.set_facecolor('#161b22')
colors = ['#00d4aa', '#f7931e']

for (name, trades), color in zip(bt.results.items(), colors):
    cum = trades['cumulative_pnl']
    ax.plot(range(len(cum)), cum, label=name, color=color, linewidth=2)

ax.axhline(0, color='#555', linewidth=0.8, linestyle='--')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))
ax.set_xlabel('Trade #', color='#aaa')
ax.set_ylabel('Cumulative P&L', color='#aaa')
ax.set_title('Equity Curve Comparison — SMA-9 vs EMA-9', color='white', fontsize=12)
ax.legend(facecolor='#222', labelcolor='white')
ax.tick_params(colors='#777')
for sp in ax.spines.values(): sp.set_edgecolor('#333')
plt.tight_layout()
plt.show()

In [ ]:
# Drawdown chart
fig, axes = plt.subplots(1, 2, figsize=(13, 4), facecolor='#0d1117')
colors = ['#00d4aa', '#f7931e']

for ax, (name, trades), color in zip(axes, bt.results.items(), colors):
    ax.set_facecolor('#161b22')
    dd = trades['drawdown']
    ax.fill_between(range(len(dd)), dd, 0, color='#e05c7f', alpha=0.5)
    ax.plot(range(len(dd)), dd, color='#e05c7f', linewidth=1)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))
    ax.set_title(f'{name} — Drawdown', color='white', fontsize=10)
    ax.tick_params(colors='#777', labelsize=7)
    for sp in ax.spines.values(): sp.set_edgecolor('#333')

fig.suptitle('Drawdown Analysis', color='white', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# P&L distribution per trade
fig, axes = plt.subplots(1, 2, figsize=(13, 4), facecolor='#0d1117')
colors = ['#00d4aa', '#f7931e']

for ax, (name, trades), color in zip(axes, bt.results.items(), colors):
    ax.set_facecolor('#161b22')
    ax.hist(trades['pnl'], bins=20, color=color, edgecolor='#333', alpha=0.85)
    ax.axvline(0, color='white', linewidth=0.8, linestyle='--')
    ax.axvline(trades['pnl'].mean(), color='#e05c7f', linewidth=1.5,
               linestyle=':', label=f"Mean: ₹{trades['pnl'].mean():,.0f}")
    ax.set_title(f'{name} — P&L Distribution', color='white', fontsize=10)
    ax.legend(facecolor='#222', labelcolor='white', fontsize=8)
    ax.tick_params(colors='#777')
    for sp in ax.spines.values(): sp.set_edgecolor('#333')

fig.suptitle('Per-Trade P&L Distribution', color='white', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Monthly P&L heatmap — SMA strategy
for name, trades in bt.results.items():
    t = trades.copy()
    t['year']  = pd.to_datetime(t['entry_date']).dt.year
    t['month'] = pd.to_datetime(t['entry_date']).dt.month
    pivot = t.pivot_table(values='pnl', index='year', columns='month', aggfunc='sum')
    pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'][:len(pivot.columns)]

    fig, ax = plt.subplots(figsize=(12, 3), facecolor='#0d1117')
    ax.set_facecolor('#161b22')
    sns.heatmap(pivot, annot=True, fmt='.0f', cmap='RdYlGn',
                center=0, ax=ax, linewidths=0.5,
                annot_kws={'size': 8}, cbar_kws={'label': 'P&L (₹)'})
    ax.set_title(f'{name} — Monthly P&L Heatmap', color='white', pad=8)
    ax.tick_params(colors='#aaa', labelsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
# Full trade log — SMA Strategy
sma_trades = bt.results['SMA-9 Strategy']
print(f"SMA-9 trades: {len(sma_trades)}")
sma_trades[['entry_date','exit_date','signal','strike','entry_spot',
             'entry_prem','exit_prem','pnl','pnl_pct','cumulative_pnl']].tail(12)

In [ ]:
# Win rate breakdown by signal type (CALL vs PUT)
for name, trades in bt.results.items():
    print(f"\n{'─'*40}")
    print(f"  {name}")
    for sig in ['CALL', 'PUT']:
        sub  = trades[trades['signal'] == sig]
        wins = (sub['pnl'] > 0).sum()
        total = len(sub)
        wr   = wins/total*100 if total else 0
        avg  = sub['pnl'].mean()
        print(f"  {sig:<5} | Count: {total:>3} | Win%: {wr:>5.1f}% | Avg P&L: ₹{avg:>8,.0f}")

## 5 · Save Outputs

In [ ]:
os.makedirs('results', exist_ok=True)

# Save trade CSVs
bt.save_trade_logs(out_dir='results')

# Save comparison chart
bt.plot_results(save_path='results/backtest_comparison.png')

# Save metrics CSV
bt.get_metrics_df().to_csv('results/metrics_summary.csv')
print('✅ All outputs saved to results/')

In [ ]:
# ── Optional: Download all results as a zip (Colab only) ──
try:
    import shutil
    from google.colab import files
    shutil.make_archive('nifty_backtest_results', 'zip', 'results')
    files.download('nifty_backtest_results.zip')
    print('📦 Download started')
except ImportError:
    print('Not in Colab — find results/ in your working directory.')